# 04 · Paper Retrieval and Recommendation

**Use Case 3: Research Paper Analysis and Recommendation System**  
**Owner:** Member C (retrieval & recommendation lead)

**Goal:** Recommend related papers for a query or a seed paper, and group papers into topics.

**Basic techniques (Weeks 1–6)**
- TF-IDF cosine similarity vs sentence-embedding similarity (e.g. `all-MiniLM-L6-v2`)
- Clustering (KMeans on embeddings) for topic discovery, with top terms per cluster

**Advanced techniques (Weeks 7–12)**
- Dense RAG retriever (embeddings + FAISS or sklearn `NearestNeighbors`) exposed as `utils.retrieve(query, k)` for member D

**Evaluation:** Precision@k / nDCG@k on test papers, counting a shared primary category as relevant; cluster purity against categories. Compare TF-IDF vs embeddings.

**Report section:** Retrieval and recommendation; system architecture diagram (~1,000 words).

## Setup

Run `01_data_loading_and_sampling.ipynb` first. Always evaluate on `test_df`.

In [ ]:
from utils import load_sample, load_split

df = load_sample()
train_df, test_df = load_split(df)
print(f"all={len(df):,}  train={len(train_df):,}  test={len(test_df):,}")
train_df[["id", "primary_category", "title"]].head()

## Imports and evaluation setup

Three document representations from Week 4 are compared for **finding similar papers**: sparse TF-IDF, averaged pretrained word vectors (spaCy `en_core_web_md`, as in the Week 4 practical) and `all-MiniLM-L6-v2` sentence embeddings (a BERT-style **encoder** model, Week 8). Retrieval index = the 9,600 training papers; queries = 200 fixed test papers, so a query is never in the index. A retrieved paper is *relevant* when it shares the query's `primary_category` (the only label we have; no citation graph exists in arXiv metadata).

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

from extraction import clean_text
from utils import save_results, ARTIFACTS_DIR

SEED = 42
N_QUERIES = 200
K = 5
CATEGORIES = sorted(df["primary_category"].unique())
results = {}

index_df = train_df.reset_index(drop=True)
query_df = test_df.sample(n=N_QUERIES, random_state=SEED).reset_index(drop=True)
index_cats = index_df["primary_category"].values
print(f"index={len(index_df):,} papers | queries={len(query_df)} | relevance = same primary category")

## Representation 1: TF-IDF (W4)

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=2, max_df=0.8, sublinear_tf=True, ngram_range=(1, 2))
t0 = time.time()
X_tfidf_index = tfidf.fit_transform(index_df["text"].map(clean_text))
X_tfidf_query = tfidf.transform(query_df["text"].map(clean_text))
print(f"TF-IDF: {X_tfidf_index.shape[1]:,} terms, {time.time() - t0:.1f} s, "
      f"density {X_tfidf_index.nnz / np.prod(X_tfidf_index.shape):.4f}")

## Representation 2: averaged word vectors (W4, practical section 4)

Each paper = mean of the 300-d GloVe-style vectors of its tokens (punctuation, spaces and out-of-vocabulary tokens skipped), exactly the `create_document_embeddings` function from the Week 4 practical. `nlp.pipe` with only the tokenizer and vectors enabled keeps this to a couple of minutes for 9,800 papers.

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
import spacy

nlp_md = spacy.load("en_core_web_md", disable=["parser", "ner", "lemmatizer", "tagger", "attribute_ruler"])
print(f"{nlp_md.meta['name']}: {nlp_md.meta['vectors']['vectors']:,} vectors, {nlp_md.meta['vectors']['width']} dims")


def averaged_word_vectors(texts):
    """Week 4 document embedding: mean of the word vectors of the tokens that have one."""
    out = np.zeros((len(texts), nlp_md.meta["vectors"]["width"]), dtype=np.float32)
    for i, doc in enumerate(nlp_md.pipe(texts, batch_size=256)):
        vectors = [t.vector for t in doc if t.has_vector and not t.is_punct and not t.is_space]
        if vectors:
            out[i] = np.mean(vectors, axis=0)
    return out


t0 = time.time()
X_w2v_index = averaged_word_vectors(index_df["text"].map(clean_text).tolist())
X_w2v_query = averaged_word_vectors(query_df["text"].map(clean_text).tolist())
print(f"word vectors: {X_w2v_index.shape}, {time.time() - t0:.0f} s")

## Representation 3: sentence embeddings (W8 encoder model)

`all-MiniLM-L6-v2` is a 6-layer BERT-style encoder fine-tuned for sentence similarity. `retrieval.embed` wraps it and L2-normalises, so cosine similarity is a dot product. The full 12k corpus is encoded once and cached (`artifacts/minilm_embeddings.npy`) because `retrieval.retrieve()` reuses the same matrix for the RAG pipeline in notebooks 05 and 06.

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
import retrieval

t0 = time.time()
all_df, all_emb = retrieval.build_index(df)            # cached after the first run
print(f"MiniLM: {all_emb.shape}, {time.time() - t0:.0f} s (0 s when cached)")

row_of = {pid: i for i, pid in enumerate(all_df["id"])}
X_st_index = all_emb[[row_of[p] for p in index_df["id"]]]
X_st_query = all_emb[[row_of[p] for p in query_df["id"]]]

## Visualising the three spaces (W4, PCA)

100 training papers per category, projected to 2-D with PCA. Tighter, better-separated colour groups mean the representation already encodes the research domain before any supervised training.

In [ ]:
viz = index_df.groupby("primary_category", group_keys=False).sample(n=100, random_state=SEED)
viz_rows = viz.index.values
spaces = {"TF-IDF": X_tfidf_index[viz_rows].toarray(),
          "Averaged word vectors": X_w2v_index[viz_rows],
          "MiniLM sentence embeddings": X_st_index[viz_rows]}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
explained = {}
for ax, (name, X) in zip(axes, spaces.items()):
    pca = PCA(n_components=2, random_state=SEED)
    pts = pca.fit_transform(X)
    explained[name] = round(float(pca.explained_variance_ratio_.sum()), 3)
    for cat in CATEGORIES:
        m = (viz["primary_category"] == cat).values
        ax.scatter(pts[m, 0], pts[m, 1], s=12, label=cat, alpha=0.7)
    ax.set_title(f"{name} (PCA, {explained[name]:.0%} variance)")
axes[0].legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()
results["pca_variance_explained"] = explained

## Cosine-similarity retrieval (W4) and evaluation: P@5 and nDCG@5

For every query paper, rank all 9,600 index papers by cosine similarity. **P@5** = share of the top-5 with the query's category; **nDCG@5** also rewards putting relevant papers higher. Random guessing gives P@5 ≈ 1/6 = 0.167.

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
def precision_at_k(ranked_cats, query_cat, k=K):
    return float(np.mean(ranked_cats[:k] == query_cat))


def ndcg_at_k(ranked_cats, query_cat, k=K):
    gains = (ranked_cats[:k] == query_cat).astype(float)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    ideal = discounts[:k].sum()                      # every top-k paper relevant
    return float((gains * discounts).sum() / ideal)


def evaluate_retriever(name, sim_matrix):
    """sim_matrix: (n_queries, n_index). Returns mean P@5 / nDCG@5 over the 200 queries."""
    p, n = [], []
    for qi, qcat in enumerate(query_df["primary_category"]):
        order = np.argsort(-sim_matrix[qi])[:K]
        ranked = index_cats[order]
        p.append(precision_at_k(ranked, qcat))
        n.append(ndcg_at_k(ranked, qcat))
    return {"retriever": name, "P@5": round(float(np.mean(p)), 3), "nDCG@5": round(float(np.mean(n)), 3)}


t0 = time.time()
sims = {
    "TF-IDF (1-2 grams)": cosine_similarity(X_tfidf_query, X_tfidf_index),
    "Averaged word vectors": cosine_similarity(X_w2v_query, X_w2v_index),
    "MiniLM sentence embeddings": X_st_query @ X_st_index.T,      # already L2-normalised
}
retrieval_table = pd.DataFrame([evaluate_retriever(n, s) for n, s in sims.items()]).set_index("retriever")
retrieval_table.loc["random baseline"] = [round(1 / len(CATEGORIES), 3), round(1 / len(CATEGORIES), 3)]
results["retrieval_eval"] = retrieval_table.reset_index().to_dict(orient="records")
print(f"{time.time() - t0:.0f} s")
retrieval_table

In [ ]:
# The same query under the three retrievers, to read by hand
qi = 0
print("QUERY:", query_df.loc[qi, "primary_category"], "|", query_df.loc[qi, "title"], "\n")
for name, s in sims.items():
    order = np.argsort(-s[qi])[:K]
    print(name)
    for r in order:
        print(f"   {s[qi, r]:.3f}  [{index_cats[r]}] {index_df.loc[r, 'title'][:90]}")
    print()

In [ ]:
# Per-category P@5 for the best retriever: which domains are easy / hard to retrieve?
best_name = retrieval_table.drop("random baseline")["nDCG@5"].idxmax()
per_cat = {}
for qi, qcat in enumerate(query_df["primary_category"]):
    order = np.argsort(-sims[best_name][qi])[:K]
    per_cat.setdefault(qcat, []).append(precision_at_k(index_cats[order], qcat))
per_cat = pd.Series({c: round(float(np.mean(v)), 3) for c, v in per_cat.items()}).sort_values()
results["best_retriever"] = best_name
results["p_at_5_by_category"] = per_cat.to_dict()
per_cat.plot.barh(title=f"P@5 by query category ({best_name})", xlim=(0, 1))
plt.show()

## Clustering (W4 text clustering): KMeans, k = 6

Unsupervised check of the same spaces: KMeans with k = 6 on the training papers. **Purity** = each cluster is credited with its majority category; **adjusted Rand index** corrects for chance. The top TF-IDF terms per MiniLM cluster show what the clusters mean.

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
def purity(labels, clusters):
    ct = pd.crosstab(clusters, labels)
    return float(ct.max(axis=1).sum() / ct.values.sum())


cluster_rows = []
cluster_assignments = {}
for name, X in {"TF-IDF (SVD-100)": None, "Averaged word vectors": X_w2v_index, "MiniLM sentence embeddings": X_st_index}.items():
    if X is None:
        from sklearn.decomposition import TruncatedSVD
        X = TruncatedSVD(n_components=100, random_state=SEED).fit_transform(X_tfidf_index)   # KMeans on 17k sparse dims is slow and poor
    km = KMeans(n_clusters=len(CATEGORIES), n_init=10, random_state=SEED).fit(X)
    cluster_assignments[name] = km.labels_
    cluster_rows.append({"space": name, "purity": round(purity(index_cats, km.labels_), 3),
                         "adjusted_rand": round(adjusted_rand_score(index_cats, km.labels_), 3)})
cluster_table = pd.DataFrame(cluster_rows).set_index("space")
results["clustering"] = cluster_table.reset_index().to_dict(orient="records")
cluster_table

In [ ]:
# What is in each MiniLM cluster: majority category + top TF-IDF terms
labels = cluster_assignments["MiniLM sentence embeddings"]
terms = tfidf.get_feature_names_out()
rows = []
for c in range(len(CATEGORIES)):
    members = np.where(labels == c)[0]
    majority = pd.Series(index_cats[members]).value_counts()
    mean_w = np.asarray(X_tfidf_index[members].mean(axis=0)).ravel()
    rows.append({"cluster": c, "papers": len(members),
                 "majority": f"{majority.index[0]} ({majority.iloc[0] / len(members):.0%})",
                 "top terms": ", ".join(terms[j] for j in np.argsort(-mean_w)[:8])})
cluster_terms = pd.DataFrame(rows).set_index("cluster")
results["minilm_cluster_terms"] = cluster_terms.reset_index().to_dict(orient="records")
pd.set_option("display.max_colwidth", 120)
cluster_terms

In [ ]:
sns.heatmap(pd.crosstab(labels, index_cats), annot=True, fmt="d", cmap="Blues")
plt.title("MiniLM KMeans clusters vs primary category")
plt.xlabel("primary category")
plt.ylabel("cluster")
plt.show()

## RAG retriever for the system (W12): `retrieval.retrieve` and `retrieval.recommend`

`codes/retrieval.py` wraps the MiniLM index over **all 12,000 papers** (not just training) so the demo can find any paper. `utils.retrieve` picks it up automatically, which is what notebooks 05 and 06 call. `recommend(paper_id)` returns the nearest papers to a given paper, excluding itself — the "recommend relevant papers" part of the use case.

> **Justification (owner, 2-4 lines in your own words):** _why this technique, which week it comes from, what it adds to the system._

In [ ]:
from utils import retrieve                      # routed to retrieval.retrieve now that retrieval.py exists

for q in ["prompt injection attacks on large language models",
          "sequential recommendation with self-attention"]:
    t0 = time.time()
    hits = retrieve(q, k=K)
    print(f"QUERY: {q}   ({(time.time() - t0) * 1000:.0f} ms)")
    for _, h in hits.iterrows():
        print(f"   {h['score']:.3f} [{h['primary_category']}] {h['title'][:90]}")
    print()

In [ ]:
seed_paper = query_df.iloc[1]
print("SEED PAPER:", f"[{seed_paper['primary_category']}]", seed_paper["title"], "\n")
recs = retrieval.recommend(seed_paper["id"], k=K)
recs[["score", "primary_category", "title"]]

In [ ]:
# Latency of the deployed retriever (what the demo and the report quote)
t0 = time.time()
for q in query_df["title"].head(50):
    retrieve(q, k=K)
results["retrieve_latency_ms"] = round((time.time() - t0) / 50 * 1000, 1)
print("mean retrieve() latency:", results["retrieve_latency_ms"], "ms (includes query encoding)")

## Findings and limitations

> **Owner, 2-4 lines each, from the numbers above:**
> - Which representation retrieves best (P@5 / nDCG@5) and by how much over TF-IDF; whether averaged word vectors sit between them and why (they lose word order and are dominated by frequent words).
> - What the PCA plots and cluster purity say: do unsupervised clusters recover the six arXiv domains, and which domains merge (compare with notebook 03's confusion pairs).
> - Limitations: same-category relevance is a proxy (a cross-listed cs.AI/cs.LG paper counts as a miss for one of them); MiniLM is trained on general text, not scientific; no citation data, so "related work" is similarity-based only.

In [ ]:
path = save_results("04", results)
print("Saved", path.relative_to(ARTIFACTS_DIR.parent))
retrieval_table